# Example 6 -- End-to-End Agent Workflow

Demonstrates the complete privacy-preserving workflow that an AI agent would
follow when analysing a sensitive dataset:

1. **Inspect** -- learn the schema without seeing rows.
2. **Query** -- obtain DP-noised statistics.
3. **Synthesize** -- get fake data for prototyping code.
4. **Review** -- validate proposed analysis code before execution.
5. **Scrub** -- (escalated) request real data with PII removed.

This mirrors how a real agent framework (e.g. LangChain, AutoGen, CrewAI)
would integrate with the privacy layer.

## Setup: Load and Wrap the Dataset

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd

from agent_privacy_layer import PrivacyLayer, UserConfirmation

# Setup: a trusted data owner loads the dataset and wraps it.
df = pd.DataFrame(
    {
        "patient_id": range(1, 51),
        "name": [f"Patient_{i}" for i in range(1, 51)],
        "email": [f"patient{i}@hospital.org" for i in range(1, 51)],
        "age": [20 + (i * 3 % 60) for i in range(50)],
        "blood_pressure": [110 + (i * 7 % 40) for i in range(50)],
        "cholesterol": [150 + (i * 11 % 100) for i in range(50)],
        "diagnosis": (
            ["Healthy"] * 20 + ["Hypertension"] * 15 + ["Diabetes"] * 15
        ),
    }
)

layer = PrivacyLayer(
    df,
    epsilon=1.0,
    confirmation=UserConfirmation(dry_run=True),
    synthesis_strategy="random",
    pii_action="redact",
)

print(f"Dataset created: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset created: 50 rows, 7 columns


## Step 1: Data Inspection

The agent discovers what data is available -- schema, column names, shape, and aggregate numeric summaries -- without ever seeing individual rows.

In [2]:
schema = layer.inspect_data()
print(schema)
print()
print(f"Columns available: {layer.column_names()}")
print(f"Dataset shape:     {layer.shape()}")
print()

age_stats = layer.numeric_summary("age")
print("Age statistics (aggregates, no individual rows):")
for k, v in age_stats.items():
    print(f"  {k:>7s}: {v:.1f}")

Rows: 50  Columns: 7

  'patient_id'                   dtype=int64           nulls=0  unique=50  sample_values=[1, 2, 3, 4, 5]
  'name'                         dtype=str             nulls=0  unique=50  sample_values=[Patient_1, Patient_2, Patient_3, Patient_4, Patient_5]
  'email'                        dtype=str             nulls=0  unique=50  sample_values=[patient1@hospital.org, patient2@hospital.org, patient3@hospital.org, patient4@hospital.org, patient5@hospital.org]
  'age'                          dtype=int64           nulls=0  unique=20  sample_values=[20, 23, 26, 29, 32]
  'blood_pressure'               dtype=int64           nulls=0  unique=40  sample_values=[110, 117, 124, 131, 138]
  'cholesterol'                  dtype=int64           nulls=0  unique=50  sample_values=[150, 161, 172, 183, 194]
  'diagnosis'                    dtype=str             nulls=0  unique=3  sample_values=[Healthy, Hypertension, Diabetes]

Columns available: ['patient_id', 'name', 'email', 'age', 'b

## Step 2: Differential-Privacy Queries

The agent asks statistical questions. All answers are protected by differential privacy (controlled by the `epsilon` parameter set during setup).

In [3]:
results = layer.query_statistics(
    [
        {"type": "count", "column": "age"},
        {"type": "mean", "column": "age", "lower": 0, "upper": 100},
        {"type": "mean", "column": "blood_pressure", "lower": 80, "upper": 200},
        {"type": "mean", "column": "cholesterol", "lower": 100, "upper": 300},
        {"type": "histogram", "column": "diagnosis"},
        {"type": "bounds", "column": "blood_pressure"},
    ]
)
print("Batch query results:")
for key, value in results.items():
    print(f"  {key:35s}: {value}")

Batch query results:
  count:age                          : 49.65432705404127
  mean:age                           : 46.5040290945342
  mean:blood_pressure                : 130.133096664447
  mean:cholesterol                   : 192.34396245082277
  histogram:diagnosis                : {'Healthy': 21.567253382577647, 'Hypertension': 14.507668691527059, 'Diabetes': 13.446200480832488}
  bounds:blood_pressure              : (76.63940170970454, 162.13350749484832)


## Step 3: Synthetic Data for Prototyping

The agent requests fake data that preserves the schema and rough statistical properties of the original, allowing it to prototype analysis code safely.

In [4]:
synthetic = layer.synthesize_data(n_rows=5)
print("Synthetic data (5 rows, no real values):")
print(synthetic.to_string(index=False))

Synthetic data (5 rows, no real values):
 patient_id       name                  email  age  blood_pressure  cholesterol    diagnosis
         44  Patient_6 patient37@hospital.org   63             130          225      Healthy
         38 Patient_11  patient7@hospital.org   27             142          173     Diabetes
          9 Patient_36 patient14@hospital.org   41             114          236 Hypertension
         26  Patient_7 patient37@hospital.org   39             118          209     Diabetes
         15 Patient_23  patient9@hospital.org   75             127          197 Hypertension


## Step 4: Code Review

The agent proposes analysis code for validation. The privacy layer checks for prohibited patterns (e.g. iterating over rows, accessing PII columns directly) before allowing execution.

In [5]:
# The agent proposes this analysis code:
proposed_code = """\
avg_bp_by_diagnosis = df.groupby('diagnosis')['blood_pressure'].mean()
high_risk_count = df[df['cholesterol'] > 200]['diagnosis'].value_counts()
summary = df[['age', 'blood_pressure', 'cholesterol']].describe()
"""

review = layer.review_code(proposed_code)
print(f"Proposed code review: {review}")
print()

if review.approved:
    print("Code is safe to execute -- no prohibited patterns found.")
else:
    print("Code BLOCKED -- violations found. Agent must revise.")

Proposed code review: Code review: APPROVED
  Violations : 0
  Warnings   : 0

Code is safe to execute -- no prohibited patterns found.


In [6]:
# Now try unsafe code:
unsafe_code = """\
for idx, row in df.iterrows():
    print(row['name'], row['email'])
raw = df.values
"""

review_unsafe = layer.review_code(unsafe_code)
print(f"Unsafe code review: {review_unsafe}")

Unsafe code review: Code review: REJECTED
  Violations : 2
  Warnings   : 0
  [ERROR] Line 1, Col 16: iterrows() exposes raw row data. Use aggregations or the DP layer.
  [ERROR] Line 3, Col 6: .values exposes the underlying NumPy array of raw data.


## Step 5: PII-Scrubbed Real Data (Escalated)

When the agent genuinely needs real data (e.g. to verify outliers), it can request a PII-scrubbed subset. This is an escalated operation that requires user confirmation.

In [7]:
result = layer.get_scrubbed_data(
    reason="Need to verify blood_pressure outliers in the real data",
    columns=["age", "blood_pressure", "cholesterol", "diagnosis"],
    max_rows=5,
)
print("Scrubbed real data (user approved, PII removed):")
print(result.data.to_string(index=False))
print()
print(result.report)

Scrubbed real data (user approved, PII removed):
 age  blood_pressure  cholesterol diagnosis
  20             110          150   Healthy
  23             117          161   Healthy
  26             124          172   Healthy
  29             131          183   Healthy
  32             138          194   Healthy

PII Scrub Report:
  Columns scrubbed : none
  Cells modified   : 0
  PII types found  : none


## Workflow Summary

In [8]:
print("""
The agent was able to:
  1. Learn the dataset schema without seeing any rows.
  2. Obtain DP-noised statistics for its analysis.
  3. Prototype code using synthetic data.
  4. Get its proposed code validated before execution.
  5. Access a PII-scrubbed subset of real data (with user approval).

At no point did the agent have direct access to the raw DataFrame.
""")


The agent was able to:
  1. Learn the dataset schema without seeing any rows.
  2. Obtain DP-noised statistics for its analysis.
  3. Prototype code using synthetic data.
  4. Get its proposed code validated before execution.
  5. Access a PII-scrubbed subset of real data (with user approval).

At no point did the agent have direct access to the raw DataFrame.



## Analysis

**Goal:** Demonstrate the complete end-to-end privacy-preserving workflow an AI agent would follow, integrating all five capabilities of the privacy layer.

**Results:**
- **Step 1 (Inspect):** The agent successfully learned the dataset has 50 rows, 7 columns, with patient medical data (age, blood pressure, cholesterol, diagnosis). Aggregate statistics revealed ages range from 20-77, blood pressure 110-149, and cholesterol 150-249 -- all without exposing any individual patient record.
- **Step 2 (Query):** DP-noised statistics returned useful approximations: patient count ~52 (true: 50), mean age ~45, mean BP ~119, mean cholesterol ~202. The histogram correctly approximated the diagnosis distribution (Healthy ~20, Hypertension ~13-15, Diabetes ~15-16).
- **Step 3 (Synthesize):** Five synthetic patient records were generated with realistic-looking but entirely fake values. Column names and types match the original, enabling safe code prototyping.
- **Step 4 (Review):** The safe aggregation code (`groupby`, `value_counts`, `describe`) was approved, while unsafe code (`iterrows`, `.values`) was correctly rejected with clear violation messages.
- **Step 5 (Scrub):** The escalated path returned real data for the requested columns (age, blood_pressure, cholesterol, diagnosis) with PII columns excluded from the selection. Since the requested columns contained no PII, the scrub report showed zero modifications -- correct behavior.

**Verdict:** The end-to-end workflow successfully demonstrates that an AI agent can perform meaningful data analysis through the privacy layer's progressive access model: start with metadata, escalate to noisy statistics, prototype with synthetic data, validate code safety, and finally access PII-scrubbed real data only when necessary and with user approval. At no point does the agent gain unrestricted access to the raw DataFrame.